In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import os
import os
import asyncio
import smtplib
import requests
from email.message import EmailMessage
from typing import Dict
from agents import Agent, Runner, function_tool
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
# 1. FIXED: Removed OpenAIChatCompletionsModel from main agents import
from agents import Agent, Runner, set_tracing_export_api_key, trace, function_tool, ModelSettings

# 2. FIXED: Imported OpenAIChatCompletionsModel from its submodule
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from typing import Dict
from IPython.display import display, Markdown
from agents import Agent, Runner, set_tracing_export_api_key, trace, function_tool, ModelSettings

In [24]:
load_dotenv()

True

In [14]:
# load env variables
load_dotenv()

set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))

# 2. Directly create the client using your GEMINI_API_KEY from .env
gemini_client = AsyncOpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 3. Pass the client to the model adapter
gemini_model = OpenAIChatCompletionsModel(
    openai_client=gemini_client,
    model="gemini-2.5-flash"
)

# declare constants
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

In [28]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and
produce a concise summary of the results. The summary must 2-3 paragraphs and les than 300 words.
Capture the main points and be succinict. Reply only with the summary.
"""

task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = []

In [22]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=gemini_model, model_settings=settings)

In [23]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))


I am sorry, but I was unable to perform a web search to provide you with the most popular AI Agent frameworks in 2026. The search tool encountered an error. Furthermore, predicting specific framework popularity for 2026 would be highly speculative even with a functional search tool, as the field of AI is evolving very rapidly.

In [36]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

# 2. FIX: Correct typing to 'list[WebSearchItem]'
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query"
    )

In [37]:
INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} searches.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=gemini_model, output_type=WebSearchPlan)

In [38]:
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='To identify the currently recognized and widely adopted AI agent frameworks, which will serve as a baseline for understanding future trends and potential popularity in 2026.', query='most popular AI agent frameworks 2023 2024'), WebSearchItem(reason='To gather expert opinions, analyst reports, and discussions regarding the projected evolution and dominant players in the AI agent framework landscape specifically towards 2026.', query='AI agent framework predictions 2025 2026 future trends'), WebSearchItem(reason='To understand the criteria and factors (e.g., scalability, ease of use, integration, community support, enterprise features) that drive the adoption and sustained popularity of AI agent frameworks.', query='factors influencing AI agent framework adoption enterprise developer community'), WebSearchItem(reason='To identify emerging technologies, research breakthroughs, or new architectural paradigms in AI agents that could lead to new

In [40]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in makrdown format, and it should be lengthy and detailed. 
Aim for 5-10 pages of content, at least 1000 words. 
"""

class ReportData(BaseModel):
    short_summary:str = Field(description="A short 2-3 sentence summary of the findings")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=gemini_model, output_type=ReportData)

In [41]:
@function_tool
def send_email_tool(subject:str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """

    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent succesfully"

In [44]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into 
 a clean, well presented HTML email with an approproate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=gemini_model)